In [39]:
from Bio import Entrez

# Always tell NCBI who you are
Entrez.email = "nathanglen@ufl.edu"

# Example accession
accession = "SRX1166121"
# accession = "SRX7636841"

# Step 1: Search SRA for the accession
handle = Entrez.esearch(db="sra", term=accession)
record = Entrez.read(handle)
handle.close()

# Step 2: Fetch summary metadata
if record["IdList"]:
    sra_id = record["IdList"][0]
    summary_handle = Entrez.esummary(db="sra", id=sra_id, rettype="json")
    summary = Entrez.read(summary_handle)
    summary_handle.close()

    print(summary)
else:
    print("No records found for", accession)


[{'Item': [], 'Id': '1696824', 'ExpXml': '<Summary><Title>Vaccine-induced immune responses after vaccination and infection with Cooperia oncophora: Sample CTRL5_7458</Title><Platform instrument_model="Illumina HiSeq 2000">ILLUMINA</Platform><Statistics total_runs="1" total_spots="17290102" total_bases="881795202" total_size="449071728" load_done="true" cluster_name="public"/></Summary><Submitter acc="SRA235633" center_name="Ghent University" contact_name="Peter Geldhof" lab_name="Veterinary parasitology"/><Experiment acc="SRX1166121" ver="1" status="public" name="Vaccine-induced immune responses after vaccination and infection with Cooperia oncophora: Sample CTRL5_7458"/><Study acc="SRP052942" name="Bos taurus breed:Holstein-Fresian Transcriptome or Gene expression"/><Organism taxid="9913" ScientificName="Bos taurus"/><Sample acc="SRS1046454" name=""/><Instrument ILLUMINA="Illumina HiSeq 2000"/><Library_descriptor><LIBRARY_NAME>CTRL5_7458</LIBRARY_NAME><LIBRARY_STRATEGY>RNA-Seq</LIBRAR

In [36]:
import xml.etree.ElementTree as ET
from textwrap import shorten

def extract_core_fields(esummary_entry):
    """
    esummary_entry: a single dict from Entrez.esummary(db="sra", ...)
    returns a small dict with the essentials
    """
    exp_xml = esummary_entry.get("ExpXml", "")
    runs_xml = esummary_entry.get("Runs", "")

    # ExpXml has multiple top-level siblings; wrap it to parse
    root = ET.fromstring(f"<root>{exp_xml}</root>")
    def txt(path):
        el = root.find(path)
        return el.text if el is not None else None
    def attr(path, name):
        el = root.find(path)
        return el.attrib.get(name) if el is not None else None

    out = {
        "experiment_acc": attr("Experiment", "acc"),
        "study_acc":      attr("Study", "acc"),
        "study_name":     attr("Study", "name"),
        "biosample":      txt("Biosample"),
        "bioproject":     txt("Bioproject"),
        "sample_acc":     attr("Sample", "acc"),
        "organism":       attr("Organism", "ScientificName"),
        "library_strategy": txt("Library_descriptor/LIBRARY_STRATEGY"),
        "library_source":   txt("Library_descriptor/LIBRARY_SOURCE"),
        "library_selection":txt("Library_descriptor/LIBRARY_SELECTION"),
        "library_layout":   ("SINGLE" if root.find("Library_descriptor/LIBRARY_LAYOUT/SINGLE") is not None
                             else "PAIRED" if root.find("Library_descriptor/LIBRARY_LAYOUT/PAIRED") is not None
                             else None),
        "protocol": txt("Library_descriptor/LIBRARY_CONSTRUCTION_PROTOCOL"),
        "title":    txt("Summary/Title"),
    }

    # Grab Run accession too (optional)
    if runs_xml:
        rroot = ET.fromstring(runs_xml)
        out["run_acc"] = rroot.attrib.get("acc")

    # Compact a human-friendly source summary
    # (shorten long protocols to one line)
    proto = (out["protocol"] or "").replace("\n", " ").strip()
    out["source_summary"] = shorten(proto, width=160, placeholder="…") or out["title"]

    return out

# --- use it on your result ---
entry = summary[0]  # from your code
info = extract_core_fields(entry)

print({
    "BioSample": info["biosample"],
    "Experiment": info["library_strategy"],
    "Organism": info["organism"],
    "Source": info["source_summary"],   # 1-line “where it came from”
    "Run": info["run_acc"],
    "Study": f'{info["study_acc"]} | {info["study_name"]}',
})


{'BioSample': 'SAMN13929375', 'Experiment': 'RNA-Seq', 'Organism': 'Macaca fascicularis', 'Source': 'Blood was collected into EDTA-containing tubes, mixed with an equal amount of PBS without calcium or magnesium, and overlaid on a 90% Ficoll-Paque Plus/10% PBS…', 'Run': 'SRR10971336', 'Study': 'SRP245407 | Contrasting Effects of Western vs. Mediterranean Diets on Monocyte Inflammatory Gene Expression and Social Behavior in a Primate Model'}


In [40]:
import xml.etree.ElementTree as ET
from typing import Any, Dict
from Bio import Entrez

def fetch_biosample_xml(samn: str) -> str:
    """
    Fetch BioSample record XML text for a given SAMN accession.
    Returns a Unicode string (decoded from bytes).
    """
    with Entrez.efetch(db="biosample", id=samn, rettype="xml", retmode="xml") as h:
        data = h.read()
    # Entrez returns bytes; decode to UTF-8 text for consistent downstream handling
    if isinstance(data, bytes):
        try:
            return data.decode("utf-8")
        except UnicodeDecodeError:
            # Fallback to latin-1 if unexpected encoding (rare)
            return data.decode("latin-1", errors="replace")
    return data

def parse_biosample_attributes(xml_txt: str) -> Dict[str, Any]:
    """
    Parse a BioSample XML (str or bytes) into a flat dict of attributes.
    Keeps ALL <Attribute attribute_name="..."> entries (lowercased keys).
    """
    out: Dict[str, Any] = {}
    try:
        # fromstring handles both str and bytes, no need for StringIO or BytesIO
        root = ET.fromstring(xml_txt)
    except ET.ParseError:
        return out

    # BioSample efetch returns <BioSampleSet><BioSample>…</BioSample></BioSampleSet>
    # Ensure we find the BioSample element even if root is already BioSample
    bs = root.find(".//BioSample") if root.tag != "BioSample" else root
    if bs is not None:
        out["biosample_accession"] = bs.attrib.get("accession")
        out["biosample_model"] = bs.attrib.get("model")

        org = bs.find(".//Organism")
        if org is not None:
            out["organism_name"] = org.attrib.get("taxonomy_name")
            out["organism_taxid"] = org.attrib.get("taxonomy_id")

        # Collect Attributes
        for attr in bs.findall(".//Attributes/Attribute"):
            name = (attr.attrib.get("attribute_name") or "").strip().lower() or "attribute"
            val = (attr.text or "").strip()
            if not val:
                continue
            key = name
            if key in out and out[key] != val:
                i = 2
                while f"{name}_{i}" in out:
                    i += 1
                key = f"{name}_{i}"
            out[key] = val

    return out

# Example single ID
xml_txt = fetch_biosample_xml("SAMN03295210")
attrs   = parse_biosample_attributes(xml_txt)
print(attrs)

{'biosample_accession': 'SAMN03295210', 'biosample_model': None, 'organism_name': 'Bos taurus', 'organism_taxid': '9913', 'sex': 'male', 'treatment': 'Control5', 'tissue': 'jejunum', 'age': '7 months', 'breed': 'Holstein-Friesian'}
